# RelCheck — GroundingDINO Detection Test**Goal:** Measure how well GroundingDINO detects entities from real COCO captions.Pipeline (mirrors actual RelCheck):1. Load COCO image + its human caption from val20142. Extract noun entities from caption (spaCy)3. Run GroundingDINO on extracted entities4. Measure: what % of entities were detected?5. Visualize bounding boxes on images**GPU:** T4 is fine. No API calls needed.

In [ ]:
!pip install -q 'transformers>=4.50.0' torch torchvision pillow pycocotools spacy!python -m spacy download en_core_web_sm -qimport torch, os, re, random, subprocess, zipfileimport numpy as npfrom PIL import Image, ImageDraw, ImageFontimport matplotlib.pyplot as pltfrom transformers import AutoProcessor, AutoModelForZeroShotObjectDetectionimport spacyDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'Device: {DEVICE}')# Load spaCy for noun extractionnlp = spacy.load('en_core_web_sm')print('✓ spaCy loaded')# Load GroundingDINOgdino_id = 'IDEA-Research/grounding-dino-tiny'gdino_proc  = AutoProcessor.from_pretrained(gdino_id)gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_id).to(DEVICE)print('✓ GroundingDINO loaded')# ThresholdsGDINO_BOX_THRESHOLD  = 0.15GDINO_TEXT_THRESHOLD  = 0.10# COCO pathsfrom google.colab import drivedrive.mount('/content/drive')COCO_ZIPS_DRIVE = '/content/drive/MyDrive/RelCheck_Data/coco_zips'COCO_IMG_ZIP    = f'{COCO_ZIPS_DRIVE}/val2014.zip'COCO_ANN_ZIP    = f'{COCO_ZIPS_DRIVE}/annotations_trainval2014.zip'COCO_DIR        = '/content/coco'COCO_IMG_DIR    = f'{COCO_DIR}/val2014'COCO_ANN_FILE   = f'{COCO_DIR}/annotations/captions_val2014.json'N_TEST = 20  # number of images to testRANDOM_SEED = 42

In [ ]:
# ── Extract COCO images + annotations from Drive ──────────os.makedirs(COCO_DIR, exist_ok=True)os.makedirs(f'{COCO_DIR}/annotations', exist_ok=True)# Unzip imagesif not os.path.isdir(COCO_IMG_DIR) or len(os.listdir(COCO_IMG_DIR)) < 100:    assert os.path.exists(COCO_IMG_ZIP), f'val2014.zip not found at {COCO_IMG_ZIP}'    print('Extracting COCO images (this takes a few minutes)...')    subprocess.run(['unzip', '-q', '-o', COCO_IMG_ZIP, '-d', COCO_DIR], check=True)    print(f'✓ Images extracted: {len(os.listdir(COCO_IMG_DIR))} files')else:    print(f'✓ COCO images already extracted: {len(os.listdir(COCO_IMG_DIR))} files')# Unzip annotationsif not os.path.exists(COCO_ANN_FILE):    assert os.path.exists(COCO_ANN_ZIP), f'annotations zip not found at {COCO_ANN_ZIP}'    print('Extracting annotations...')    subprocess.run(['unzip', '-q', '-o', COCO_ANN_ZIP, '-d', COCO_DIR], check=True)    print('✓ Annotations extracted')else:    print('✓ Annotations already present')# ── Load COCO captions ────────────────────────────────────from pycocotools.coco import COCOcoco_caps = COCO(COCO_ANN_FILE)coco_img_ids = coco_caps.getImgIds()random.seed(RANDOM_SEED)random.shuffle(coco_img_ids)# Select N_TEST images that have captions and exist on diskcoco_data = {}for cid in coco_img_ids:    if len(coco_data) >= N_TEST:        break    ann_ids = coco_caps.getAnnIds(imgIds=cid)    anns = coco_caps.loadAnns(ann_ids)    gt_caps = [a['caption'].strip() for a in anns if a['caption'].strip()]    if len(gt_caps) < 1:        continue    img_info = coco_caps.loadImgs(cid)[0]    img_path = os.path.join(COCO_IMG_DIR, img_info['file_name'])    if not os.path.exists(img_path):        continue    coco_data[cid] = {        'path': img_path,        'file_name': img_info['file_name'],        'gt_captions': gt_caps,    }print(f'\n✓ Loaded {len(coco_data)} COCO images with captions')for cid in list(coco_data.keys())[:3]:    d = coco_data[cid]    print(f'  COCO {cid}: "{d["gt_captions"][0][:80]}..."')

In [ ]:
# ── Entity extraction using spaCy ──────────────────────────def extract_entities(caption: str) -> list:    '''Extract noun entities from caption using spaCy NER + noun chunks.'''    doc = nlp(caption)        entities = []        # Get noun chunks (e.g., "a large brown dog", "the wooden table")    for chunk in doc.noun_chunks:        # Strip determiners (a, the, an) but keep adjectives        tokens = [t for t in chunk if t.pos_ != 'DET']        if tokens:            entity = ' '.join(t.text.lower() for t in tokens)            entities.append(entity)        # Deduplicate while preserving order    seen = set()    unique = []    for e in entities:        if e not in seen:            seen.add(e)            unique.append(e)        return unique# ── GroundingDINO detection (batched) ─────────────────────def run_gdino(img: Image.Image, queries: list, batch_size: int = 4) -> dict:    '''Run GDINO in batches, return {query: [{box, score, label}]}.'''    if not queries:        return {}        clean_queries = list(dict.fromkeys(q.strip().lower() for q in queries if q.strip()))        # Add articles    article_queries = []    article_to_query = {}    for q in clean_queries:        if q.split()[0] in ('a', 'an', 'the'):            aq = q        else:            aq = f'a {q}'        article_queries.append(aq)        article_to_query[aq] = q        results = {q: [] for q in clean_queries}        for batch_start in range(0, len(article_queries), batch_size):        batch = article_queries[batch_start:batch_start + batch_size]        batch_map = {aq: article_to_query[aq] for aq in batch}                inputs = gdino_proc(            images=img,            text=[batch],            return_tensors='pt',        ).to(DEVICE)                with torch.no_grad():            outputs = gdino_model(**inputs)                results_list = gdino_proc.post_process_grounded_object_detection(            outputs,            inputs.input_ids,            threshold=GDINO_BOX_THRESHOLD,            text_threshold=GDINO_TEXT_THRESHOLD,            target_sizes=[(img.height, img.width)],        )                raw = results_list[0]        boxes  = raw['boxes'].cpu().numpy()        scores = raw['scores'].cpu().numpy()                if 'text_labels' in raw:            labels = raw['text_labels']        else:            labels = raw.get('labels', [])            if hasattr(labels, 'cpu'):                labels = labels.cpu().tolist()                for box, score, label in zip(boxes, scores, labels):            x1, y1, x2, y2 = box.tolist()            x1, y1 = max(0, int(x1)), max(0, int(y1))            x2, y2 = min(img.width, int(x2)), min(img.height, int(y2))                        label_str = (label if isinstance(label, str) else str(label)).strip().lower()                        matched_query = batch_map.get(label_str)            if matched_query is None:                stripped = re.sub(r'^(a|an|the)\s+', '', label_str)                for aq, oq in batch_map.items():                    if oq == stripped or oq in stripped or stripped in oq:                        matched_query = oq                        break            if matched_query is None:                for q in [article_to_query[aq] for aq in batch]:                    if q in label_str or label_str in q:                        matched_query = q                        break            if matched_query is None:                matched_query = label_str                        if matched_query not in results:                results[matched_query] = []                        results[matched_query].append({                'box': [x1, y1, x2, y2],                'score': float(score),                'label': matched_query,            })        return results# ── Draw bounding boxes ───────────────────────────────────COLORS = ['#FF0000', '#0066FF', '#00CC00', '#FF9900', '#9933FF',          '#FF3399', '#00CCCC', '#996633', '#66FF00', '#FF6666']def draw_boxes(img, detections_dict):    '''Draw bounding boxes on image. Returns annotated PIL image.'''    annotated = img.copy()    draw = ImageDraw.Draw(annotated)    ci = 0    for query, dets in detections_dict.items():        color = COLORS[ci % len(COLORS)]        ci += 1        for d in dets:            x1, y1, x2, y2 = d['box']            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)            label = f"{query} ({d['score']:.2f})"            # Background for text            draw.rectangle([x1, max(0, y1-14), x1+len(label)*7, y1], fill=color)            draw.text((x1+2, max(0, y1-13)), label, fill='white')    return annotatedprint('✓ Functions defined: extract_entities, run_gdino, draw_boxes')

In [ ]:
# ── Run the actual test ────────────────────────────────────print('='*70)print(f'GROUNDING DINO DETECTION TEST — {len(coco_data)} COCO images')print('='*70)all_results = []for idx, (cid, data) in enumerate(coco_data.items()):    # Load image    img = Image.open(data['path']).convert('RGB')        # Use first GT caption    caption = data['gt_captions'][0]        # Extract entities from caption (like the real pipeline does)    entities = extract_entities(caption)        if not entities:        print(f'[{idx+1}/{len(coco_data)}] COCO {cid}: no entities extracted from "{caption[:60]}..."')        continue        # Run GDINO    detections = run_gdino(img, entities)        # Count    detected = {q: d for q, d in detections.items() if d}    missed   = [q for q in entities if not detections.get(q)]    rate     = len(detected) / len(entities) if entities else 0        all_results.append({        'coco_id': cid,        'caption': caption,        'entities': entities,        'detected': list(detected.keys()),        'missed': missed,        'rate': rate,        'img': img,        'detections': detections,    })        status = '✓' if rate >= 0.5 else '✗'    print(f'[{idx+1}/{len(coco_data)}] {status} COCO {cid}: {len(detected)}/{len(entities)} entities detected ({rate*100:.0f}%)')    if missed:        print(f'         Missed: {missed}')# ── Summary metrics ───────────────────────────────────────print('\n' + '='*70)print('SUMMARY')print('='*70)total_entities = sum(len(r['entities']) for r in all_results)total_detected = sum(len(r['detected']) for r in all_results)total_missed   = sum(len(r['missed']) for r in all_results)overall_rate = total_detected / total_entities if total_entities else 0per_image_rates = [r['rate'] for r in all_results]full_detect = sum(1 for r in all_results if r['rate'] == 1.0)zero_detect = sum(1 for r in all_results if r['rate'] == 0.0)print(f'Images tested:        {len(all_results)}')print(f'Total entities:       {total_entities}')print(f'Total detected:       {total_detected} ({overall_rate*100:.1f}%)')print(f'Total missed:         {total_missed}')print(f'Mean per-image rate:  {np.mean(per_image_rates)*100:.1f}%')print(f'Median per-image rate:{np.median(per_image_rates)*100:.1f}%')print(f'All entities found:   {full_detect}/{len(all_results)} images')print(f'Zero entities found:  {zero_detect}/{len(all_results)} images')# Most commonly missed entity typesfrom collections import Countermissed_counts = Counter()for r in all_results:    for m in r['missed']:        # Get head noun        missed_counts[m] += 1print(f'\nMost commonly missed entities:')for entity, count in missed_counts.most_common(10):    print(f'  {entity:25s}: missed {count} time(s)')

In [ ]:
# ── Visualize: show images with bounding boxes ────────────print('='*70)print('VISUALIZATIONS')print('='*70)# Show up to 10 images in a gridn_show = min(10, len(all_results))cols = 2rows = (n_show + 1) // 2fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))axes = axes.flatten() if n_show > 2 else [axes] if n_show == 1 else axes.flatten()for i in range(n_show):    r = all_results[i]    ax = axes[i]        # Draw boxes for detected entities    found = {q: d for q, d in r['detections'].items() if d}    annotated = draw_boxes(r['img'], found)        ax.imshow(annotated)        # Title: caption + detection rate    title = f"COCO {r['coco_id']} — {len(r['detected'])}/{len(r['entities'])} detected"    if r['missed']:        title += f"\nMissed: {', '.join(r['missed'][:3])}"    ax.set_title(title, fontsize=9, wrap=True)    ax.axis('off')# Hide unused axesfor j in range(n_show, len(axes)):    axes[j].axis('off')plt.suptitle(f'GroundingDINO Detection: {total_detected}/{total_entities} entities ({overall_rate*100:.1f}%)',             fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# ── Detailed per-image table ───────────────────────────────print('='*70)print('DETAILED RESULTS')print('='*70)print(f'\n{"COCO ID":<12s} {"Rate":>6s}  {"Caption":<55s}  {"Detected":<25s}  {"Missed"}')print('-'*140)for r in all_results:    cap_short = r['caption'][:52] + '...' if len(r['caption']) > 55 else r['caption']    det_str = ', '.join(r['detected'][:3])    if len(r['detected']) > 3:        det_str += f' +{len(r["detected"])-3}'    miss_str = ', '.join(r['missed'][:3])    if len(r['missed']) > 3:        miss_str += f' +{len(r["missed"])-3}'        rate_pct = f"{r['rate']*100:.0f}%"    print(f"{r['coco_id']:<12d} {rate_pct:>6s}  {cap_short:<55s}  {det_str:<25s}  {miss_str}")